# Model Tuning & Testing Notebook

Offline hyperparameter tuning and testing for XGBoost inference models.

**Approach:**
- Hold out `season ∈ {2025, 2026}` as a final test set
- CV on the remaining seasons: leave-one-season-out (LOSO) as target only
- Hyperparameter selection criterion: mean LOSO CV score
- Test-set metrics are final error estimates

Targets: `ev_atoi`, `pp_atoi`, `pk_atoi`, `gp_rate`, `evg60`, `eva60`, `ppg60`, `ppa60`

## Imports

In [130]:
import os, sys, itertools, time
import numpy as np
import pandas as pd
import xgboost as xgb

In [131]:
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
from feature_engineering import (
    ALL_TARGETS, FEATURE_SETS_XGB,
    build_modeling_frame, build_inference_frame, training_filter, compute_sample_weights,
)

## Setup

In [132]:
TEST_SEASONS = [2025, 2026]
PROJECTION_YEAR = 2027

In [133]:
frame = build_modeling_frame()
print('Frame dimensions:', frame.shape)

Frame dimensions: (14509, 65)


In [134]:
print('Rows per season\n\n', frame.groupby('season').size(), sep='')

Rows per season

season
2011     891
2012     894
2013     839
2014     886
2015     882
2016     898
2017     888
2018     890
2019     906
2020     883
2021     913
2022    1004
2023     951
2024     924
2025     920
2026     940
dtype: int64


In [135]:
frame

,playerId,skaterFullName,season,positionCode,ev_atoi,pp_atoi,pk_atoi,gp_rate,evg60,eva60,...,lag3_pp_toi_sec,lag3_pk_toi_sec,skaterFullName_bio,positionCode_bio,age,age2,age3,draft_round,draft_overall,is_rookie
0,8467875,Daniel Sedin,2011,L,14.838415,3.609950,0.104667,1.000000,1.134169,1.923156,...,20678.0,NaN,Daniel Sedin,L,30.012320,900.739372,27033.278550,1.0,2.0,0
1,8466378,Martin St. Louis,2011,R,15.999187,4.516867,0.463817,1.000000,1.234819,1.417755,...,20553.0,NaN,Martin St. Louis,R,35.288159,1245.254151,43942.726227,8.0,225.0,0
2,8470621,Corey Perry,2011,R,17.159553,3.508117,1.645317,1.000000,1.364525,1.279242,...,14628.0,NaN,Corey Perry,R,25.377139,643.999181,16342.856699,1.0,28.0,0
3,8467876,Henrik Sedin,2011,C,15.510976,3.595517,0.153033,1.000000,0.518909,2.264329,...,20703.0,NaN,Henrik Sedin,C,30.012320,900.739372,27033.278550,1.0,3.0,0
4,8474564,Steven Stamkos,2011,C,15.322967,4.547350,0.323983,1.000000,1.337065,1.289313,...,NaN,NaN,Steven Stamkos,C,20.646133,426.262799,8800.678350,1.0,1.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14504,8484821,Sam Morton,2026,C,8.038889,0.044433,1.250000,0.036585,0.000000,0.000000,...,NaN,NaN,Sam Morton,C,26.179329,685.357279,17942.193838,8.0,225.0,0
14505,8485251,Kai Uchacz,2026,C,11.516667,0.405550,0.261100,0.036585,0.000000,0.000000,...,NaN,NaN,Kai Uchacz,C,22.272416,496.060521,11048.466368,8.0,225.0,1
14506,8485385,Braeden Cootes,2026,C,9.638889,1.138883,0.000000,0.036585,0.000000,0.000000,...,NaN,NaN,Braeden Cootes,C,18.642026,347.525134,6478.572582,1.0,15.0,1
14507,8486166,Viking Gustafsson Nyberg,2026,D,17.066667,0.000000,4.225000,0.024390,0.000000,0.000000,...,NaN,NaN,Viking Gustafsson Nyberg,D,22.028747,485.265713,10689.795840,8.0,225.0,1


In [136]:
frame.columns

Index(['playerId', 'skaterFullName', 'season', 'positionCode', 'ev_atoi',
       'pp_atoi', 'pk_atoi', 'gp_rate', 'evg60', 'eva60', 'ppg60', 'ppa60',
       'gp', 'ev_toi_sec', 'pp_toi_sec', 'pk_toi_sec', 'is_defense',
       'is_forward', 'lag1_gp', 'lag1_gp_rate', 'lag1_all_atoi',
       'lag1_ev_atoi', 'lag1_pp_atoi', 'lag1_pk_atoi', 'lag1_evg60',
       'lag1_eva60', 'lag1_ppg60', 'lag1_ppa60', 'lag1_ev_toi_sec',
       'lag1_pp_toi_sec', 'lag1_pk_toi_sec', 'lag2_gp', 'lag2_gp_rate',
       'lag2_all_atoi', 'lag2_ev_atoi', 'lag2_pp_atoi', 'lag2_pk_atoi',
       'lag2_evg60', 'lag2_eva60', 'lag2_ppg60', 'lag2_ppa60',
       'lag2_ev_toi_sec', 'lag2_pp_toi_sec', 'lag2_pk_toi_sec', 'lag3_gp',
       'lag3_gp_rate', 'lag3_all_atoi', 'lag3_ev_atoi', 'lag3_pp_atoi',
       'lag3_pk_atoi', 'lag3_evg60', 'lag3_eva60', 'lag3_ppg60', 'lag3_ppa60',
       'lag3_ev_toi_sec', 'lag3_pp_toi_sec', 'lag3_pk_toi_sec',
       'skaterFullName_bio', 'positionCode_bio', 'age', 'age2', 'age3',
       'dr

## Definitions

In [137]:
DEFAULT_XGB_PARAMS = dict(
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    tree_method='hist', objective='reg:squarederror', n_jobs=4, verbosity=0,
)

In [138]:
# Fit XGB regressgor on train_sub for given target and hyperparams
def fit_xgb(target, train_sub, hp):
    feats = FEATURE_SETS_XGB[target]
    X = train_sub[feats].values
    y = train_sub[target].astype(float).values
    w = compute_sample_weights(train_sub, target, hp['decay'], PROJECTION_YEAR)
    params = {
        **DEFAULT_XGB_PARAMS,
        'n_estimators': hp.get('n_estimators', 200),
        'max_depth': hp.get('max_depth', 3),
        'learning_rate': hp.get('learning_rate', 0.05),
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X, y, sample_weight=w)
    return model

In [139]:
# Predict with fitted XGB model on eval_sub for given target
def predict_xgb(model, target, eval_sub):
    feats = FEATURE_SETS_XGB[target]
    return model.predict(eval_sub[feats].values)

In [140]:
# Compute error metrics between true and predicted target values
def metrics(y_true, y_pred):
    err = y_true - y_pred
    return {
        'mae':  float(np.mean(np.abs(err))),
        'rmse': float(np.sqrt(np.mean(err ** 2))),
        'r2':   float(1 - np.var(err) / max(np.var(y_true), 1e-9)),
    }

In [141]:
# LOSO CV. Returns list of (season, rmse) tuples, one per held-out fold.
def cv_fold_scores(target, train_frame, hp):
    seasons = sorted(train_frame['season'].unique())
    fold_scores = []
    for held in seasons:
        tr = train_frame[train_frame['season'] != held]
        ev = train_frame[train_frame['season'] == held]
        ev = ev[training_filter(ev, target)]
        if ev.empty or tr.empty:
            continue
        model = fit_xgb(target, tr, hp)
        preds = predict_xgb(model, target, ev)
        rmse = metrics(ev[target].astype(float).values, preds)['rmse']
        fold_scores.append((held, rmse))
    return fold_scores

In [142]:
# Train on train_frame, predict on test_frame. Return metrics and predictions df
def held_out_test(target, hp, train_frame, test_frame):
    feats = FEATURE_SETS_XGB[target]
    test_filt = test_frame[training_filter(test_frame, target)].copy()
    model = fit_xgb(target, train_frame, hp)
    preds = predict_xgb(model, target, test_filt)
    y_true = test_filt[target].astype(float).values
    m = metrics(y_true, preds)

    meta_cols = ['playerId', 'skaterFullName', 'season', 'positionCode', 'age']
    available_meta = [c for c in meta_cols if c in test_filt.columns]
    pred_df = test_filt[available_meta + [f for f in feats if f not in available_meta]].copy()
    pred_df[f'{target}_true'] = y_true
    pred_df[f'{target}_pred'] = preds
    pred_df[f'{target}_error'] = y_true - preds
    pred_df[f'{target}_abs_error'] = np.abs(pred_df[f'{target}_error'])
    pred_df = pred_df.sort_values(f'{target}_abs_error', ascending=False).reset_index(drop=True)

    return m, pred_df

## Tuning Loop

Grid search over XGBoost hyperparams. Selection criterion is minimum mean LOSO CV RMSE.

In [143]:
XGB_GRID = {
    'n_estimators': [100, 200, 400],
    'max_depth': [3, 4],
    'learning_rate': [0.03, 0.05, 0.10],
    'decay': [0.0, 0.05, 0.10],
}

In [144]:
def grid_iter(grid):
    keys = list(grid.keys())
    for combo in itertools.product(*[grid[k] for k in keys]):
        yield dict(zip(keys, combo))

In [145]:
grid_size = len(list(grid_iter(XGB_GRID)))
print(f'Grid size: {grid_size} combinations per target ({grid_size * len(ALL_TARGETS)} total fits)')

Grid size: 54 combinations per target (432 total fits)


In [146]:
raise NotImplementedError('WIP')

NotImplementedError: WIP

In [ ]:
results = []
for t_idx, target in enumerate(ALL_TARGETS):
    base = frame[training_filter(frame, target)]
    train_pool = base[~base['season'].isin(TEST_SEASONS)]
    test_pool  = base[base['season'].isin(TEST_SEASONS)]

    best_hp = None
    best_cv = float('inf')
    t0 = time.time()

    for k_idx, hp in enumerate(grid_iter(XGB_GRID)):
        fold_scores = cv_fold_scores(target, train_pool, hp)
        cv_mean = float(np.mean([s for _, s in fold_scores])) if fold_scores else float('nan')
        print(f'  [{target} {t_idx+1}/{len(ALL_TARGETS)}] combo {k_idx+1}/{grid_size}  '
              f'n_est={hp["n_estimators"]} depth={hp["max_depth"]} lr={hp["learning_rate"]} '
              f'decay={hp["decay"]}  cv_rmse={cv_mean:.4f}')
        if cv_mean < best_cv:
            best_cv = cv_mean
            best_hp = hp

    test_metrics, _ = held_out_test(target, best_hp, train_pool, test_pool)
    elapsed = time.time() - t0
    print(f'\n[{target}] DONE in {elapsed:.1f}s | best CV RMSE={best_cv:.4f} '
          f'(n_est={best_hp["n_estimators"]} depth={best_hp["max_depth"]} '
          f'lr={best_hp["learning_rate"]} decay={best_hp["decay"]}) '
          f'| test RMSE={test_metrics["rmse"]:.4f} MAE={test_metrics["mae"]:.4f} R²={test_metrics["r2"]:.4f}\n')

    results.append({
        'target': target,
        'best_hp': best_hp,
        'cv_rmse': best_cv,
        'test_rmse': test_metrics['rmse'],
        'test_mae': test_metrics['mae'],
        'test_r2': test_metrics['r2'],
    })

results_df = pd.DataFrame(results)
results_df

## `MODEL_CONFIG` block

Copy-paste into `model_training.py`.

In [ ]:
lines = ["MODEL_CONFIG = {"]
for r in results:
    hp = r['best_hp']
    xgb_p = {k: hp[k] for k in ('n_estimators', 'max_depth', 'learning_rate')}
    lines.append(f"    '{r['target']}': {{'decay': {hp['decay']}, 'xgb_params': {xgb_p}}},")
lines.append('}')
print('\n'.join(lines))

---
## Testing

Evaluates a specific XGBoost configuration and outputs the following:
- Per-fold CV scores
- Mean CV score
- Test-set score (retrained on full train set)
- Predictions dataframe with features, true values, and errors

### Configuration

In [148]:
TEST_TARGET = 'evg60'
TEST_HP = {
    'n_estimators': 200,
    'max_depth': 3,
    'learning_rate': 0.05,
    'decay': 0.10,
}

### Per-fold CV scores

In [149]:
base = frame[training_filter(frame, TEST_TARGET)]
train_pool = base[~base['season'].isin(TEST_SEASONS)]
test_pool  = base[base['season'].isin(TEST_SEASONS)]

fold_scores = cv_fold_scores(TEST_TARGET, train_pool, TEST_HP)

fold_df = pd.DataFrame(fold_scores, columns=['season', 'rmse'])
print(f'Target: {TEST_TARGET}')
print(fold_df.to_string(index=False))
print(f'\nMean CV RMSE: {fold_df["rmse"].mean():.4f}')
print(f'Std  CV RMSE: {fold_df["rmse"].std():.4f}')

Target: evg60
 season     rmse
   2011 0.222477
   2012 0.233267
   2013 0.254715
   2014 0.234999
   2015 0.236993
   2016 0.231352
   2017 0.251159
   2018 0.227953
   2019 0.246688
   2020 0.244878
   2021 0.271923
   2022 0.261720
   2023 0.256644
   2024 0.249862

Mean CV RMSE: 0.2446
Std  CV RMSE: 0.0141


### Test-set evaluation (retrained on full train set)

In [150]:
test_m, pred_df = held_out_test(TEST_TARGET, TEST_HP, train_pool, test_pool)

print(f'Target: {TEST_TARGET} | Test seasons: {TEST_SEASONS}')
print(f'  RMSE: {test_m["rmse"]:.4f}')
print(f'  MAE:  {test_m["mae"]:.4f}')
print(f'  R²:   {test_m["r2"]:.4f}')
print(f'  N:    {len(pred_df)}')

Target: evg60 | Test seasons: [2025, 2026]
  RMSE: 0.2421
  MAE:  0.1832
  R²:   0.5968
  N:    1427


### Test predictions

In [151]:
pred_df

,playerId,skaterFullName,season,positionCode,age,age2,age3,is_defense,draft_round,draft_overall,lag1_evg60,lag2_evg60,lag3_evg60,lag1_ev_atoi,lag1_eva60,lag1_ev_toi_sec,evg60_true,evg60_pred,evg60_error,evg60_abs_error
0,8473419,Brad Marchand,2026,L,37.390828,1398.074033,52275.145996,0,3.0,71.0,1.098454,0.991937,0.724383,13.847887,1.159479,58992.0,1.865147,0.754245,1.110902,1.110902
1,8471214,Alex Ovechkin,2025,L,39.039014,1524.044643,59497.200735,0,1.0,1.0,0.932267,1.487537,1.627174,14.664135,0.984059,69508.0,2.042862,1.036167,1.006695,1.006695
2,8480448,Parker Kelly,2026,C,26.384668,696.150707,18367.705316,0,8.0,225.0,0.487635,0.478289,0.132602,10.766250,0.696621,51678.0,1.386989,0.531614,0.855376,0.855376
3,8483445,Cutter Gauthier,2026,L,21.700205,470.898912,10218.603079,0,1.0,5.0,1.053008,0.000000,NaN,12.507723,1.170009,61538.0,1.675198,0.825443,0.849754,0.849754
4,8482699,Dylan Guenther,2026,R,22.477755,505.249468,11356.873740,0,1.0,9.0,0.958381,0.862198,0.326264,13.415476,1.022273,56345.0,1.696693,0.865724,0.830969,0.830969
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1422,8481059,Scott Perunovich,2025,D,26.121834,682.350230,17824.239692,1,2.0,45.0,0.000000,NaN,0.000000,13.066667,0.935374,42336.0,0.229856,0.230189,-0.000333,0.000333
1423,8483490,Pavel Mintyukov,2025,D,20.851472,434.783868,9065.883467,1,1.0,10.0,0.238846,NaN,NaN,15.949735,0.955382,60290.0,0.278302,0.278024,0.000278,0.000278
1424,8476467,Jamie Oleksiak,2025,D,31.778234,1009.856162,32091.445498,1,1.0,14.0,0.087915,0.441062,0.047681,16.645732,0.483534,81897.0,0.182145,0.181996,0.000149,0.000149
1425,8483565,Nick Blankenburg,2025,D,26.390144,696.439686,18379.143430,1,8.0,225.0,0.316039,0.397395,0.000000,15.820833,0.000000,11391.0,0.185886,0.185797,0.000089,0.000089


In [152]:
pred_df.columns

Index(['playerId', 'skaterFullName', 'season', 'positionCode', 'age', 'age2',
       'age3', 'is_defense', 'draft_round', 'draft_overall', 'lag1_evg60',
       'lag2_evg60', 'lag3_evg60', 'lag1_ev_atoi', 'lag1_eva60',
       'lag1_ev_toi_sec', 'evg60_true', 'evg60_pred', 'evg60_error',
       'evg60_abs_error'],
      dtype='object')

In [153]:
pred_df[['playerId', 'skaterFullName', 'season', 'positionCode', 'age',
          f'lag1_{TEST_TARGET}', f'lag2_{TEST_TARGET}', f'lag3_{TEST_TARGET}',
          f'{TEST_TARGET}_true', f'{TEST_TARGET}_pred', f'{TEST_TARGET}_error']].head(25)

,playerId,skaterFullName,season,positionCode,age,lag1_evg60,lag2_evg60,lag3_evg60,evg60_true,evg60_pred,evg60_error
0,8473419,Brad Marchand,2026,L,37.390828,1.098454,0.991937,0.724383,1.865147,0.754245,1.110902
1,8471214,Alex Ovechkin,2025,L,39.039014,0.932267,1.487537,1.627174,2.042862,1.036167,1.006695
2,8480448,Parker Kelly,2026,C,26.384668,0.487635,0.478289,0.132602,1.386989,0.531614,0.855376
3,8483445,Cutter Gauthier,2026,L,21.700205,1.053008,0.000000,NaN,1.675198,0.825443,0.849754
4,8482699,Dylan Guenther,2026,R,22.477755,0.958381,0.862198,0.326264,1.696693,0.865724,0.830969
5,8481656,Aliaksei Protas,2025,L,23.734428,0.364883,0.369307,0.503332,1.445396,0.614770,0.830626
6,8481540,Cole Caufield,2026,R,24.744695,1.319469,0.878342,1.677169,2.001056,1.185574,0.815482
7,8478874,Adam Gaudette,2025,R,27.994524,0.000000,NaN,0.305205,1.322162,0.506803,0.815359
8,8477380,Jonny Brodzinski,2025,C,31.285421,0.510609,0.394694,0.274140,1.287592,0.485257,0.802335
9,8477934,Leon Draisaitl,2025,C,28.930869,0.883273,0.833161,1.280805,1.673035,0.892319,0.780717


In [154]:
pred_df[[f'{TEST_TARGET}_true', f'{TEST_TARGET}_pred',
         f'{TEST_TARGET}_error', f'{TEST_TARGET}_abs_error']].describe()

,evg60_true,evg60_pred,evg60_error,evg60_abs_error
count,1427.000000,1427.000000,1427.000000,1427.000000
mean,0.542607,0.548681,-0.006074,0.183175
std,0.381241,0.296467,0.242095,0.158336
min,0.000000,0.094905,-0.756729,0.000041
25%,0.226649,0.223415,-0.156225,0.065467
50%,0.495011,0.598018,-0.016498,0.137502
75%,0.795257,0.760629,0.117990,0.260287
max,2.042862,1.427043,1.110902,1.110902


### Future predictions

In [158]:
inference_frame = build_inference_frame(PROJECTION_YEAR)
inference_frame['skaterFullName'] = inference_frame['skaterFullName'].fillna(inference_frame['skaterFullName_bio'])

raw_seasons = build_modeling_frame()[['playerId', 'season']]
active_ids = raw_seasons.loc[raw_seasons['season'].isin([PROJECTION_YEAR - 1, PROJECTION_YEAR]), 'playerId'].unique()
inference_frame = inference_frame[inference_frame['playerId'].isin(active_ids)].reset_index(drop=True)

print(f'{PROJECTION_YEAR} inference frame: {inference_frame.shape[0]} candidates')
inference_frame[['playerId', 'skaterFullName', 'positionCode', 'age', f'lag1_{TEST_TARGET}', f'lag2_{TEST_TARGET}', f'lag3_{TEST_TARGET}']].head()

2027 inference frame: 940 candidates


,playerId,skaterFullName,positionCode,age,lag1_evg60,lag2_evg60,lag3_evg60
0,8470613,Brent Burns,D,41.563313,0.549178,0.259931,0.220794
1,8470621,Corey Perry,R,41.377139,0.881777,1.045418,0.868284
2,8471214,Alex Ovechkin,L,41.037645,1.526933,2.042862,0.932267
3,8471215,Evgeni Malkin,C,40.169747,1.113494,0.692974,1.014711
4,8471675,Sidney Crosby,C,39.151266,1.041429,0.918111,1.426802


In [159]:
full_pool = frame[training_filter(frame, TEST_TARGET)]
full_model = fit_xgb(TEST_TARGET, full_pool, TEST_HP)

future_preds = predict_xgb(full_model, TEST_TARGET, inference_frame)
future_df = inference_frame[['playerId', 'skaterFullName', 'positionCode', 'age', f'lag1_{TEST_TARGET}', f'lag2_{TEST_TARGET}', f'lag3_{TEST_TARGET}']].copy()
future_df[f'{TEST_TARGET}_pred'] = future_preds
future_df = future_df.sort_values(f'{TEST_TARGET}_pred', ascending=False).reset_index(drop=True)

print(f'Target: {TEST_TARGET} | Projection year: {PROJECTION_YEAR} | N={len(future_df)}')
future_df.head(25)

Target: evg60 | Projection year: 2027 | N=940


,playerId,skaterFullName,positionCode,age,lag1_evg60,lag2_evg60,lag3_evg60,evg60_pred
0,8479318,Auston Matthews,C,29.037645,1.304219,1.185622,2.242634,1.301096
1,8479420,Tage Thompson,C,28.919918,1.605435,1.852006,1.048221,1.274152
2,8477939,William Nylander,R,30.417522,1.284717,1.483794,1.224137,1.270107
3,8477492,Nathan MacKinnon,C,31.082820,1.757506,0.924468,1.643964,1.259657
4,8478398,Kyle Connor,L,29.809719,1.349215,1.309837,1.648274,1.253638
5,8477960,Adrian Kempe,R,30.047912,1.510126,1.336174,1.101895,1.237179
6,8478864,Kirill Kaprizov,L,29.431896,1.103045,1.603666,1.243110,1.228638
7,8479542,Brandon Hagel,L,28.095825,1.653247,1.445130,1.093616,1.226594
8,8484801,Macklin Celebrini,C,20.301164,1.550117,0.885750,NaN,1.204783
9,8476887,Filip Forsberg,L,32.134155,1.339269,0.995969,1.668830,1.197156
